# SHAP Analysis — 200 Stratified Instances (Kaggle GPU Notebook)
Run on Kaggle with GPU accelerator.
Upload the 6 files from kaggle_upload/ as a dataset named 'shap-irrigation-data'.

Outputs (download these when done):
  - shap_values_200.npy          (200, 15, 6) raw SHAP values
  - shap_indices_200.npy         indices into X_test
  - bootstrap_feature_importance_200.npy   (1000, 6)
  - bootstrap_temporal_importance_200.npy  (1000, 15, 6)
  - shap_feature_importance_200.csv
  - shap_temporal_importance_200.csv
  - shap_importance_with_ci_200.csv
  - convergence_results_200.json
  - shap_importance_with_ci.png
  - temporal_importance_heatmap.png
  - shap_waterfall_plots.png

## 0. Install & Import

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "shap", "-q"])

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
import time
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import shap

print(f"TF {tf.__version__}, SHAP {shap.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

## 1. AttentionLayer (must match training code exactly)

In [ ]:
class AttentionLayer(layers.Layer):
    def __init__(self, units=20, return_sequences=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], self.units),
            initializer='glorot_uniform', trainable=True, name='attention_weight')
        self.b = self.add_weight(shape=(self.units,),
            initializer='zeros', trainable=True, name='attention_bias')
        self.u = self.add_weight(shape=(self.units,),
            initializer='glorot_uniform', trainable=True, name='attention_vector')
        super().build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        attention_weights = tf.nn.softmax(tf.tensordot(score, self.u, axes=1), axis=1)
        attention_weights = tf.expand_dims(attention_weights, axis=-1)
        context_vector = tf.reduce_sum(inputs * attention_weights, axis=1)
        if self.return_sequences:
            return context_vector, tf.squeeze(attention_weights, axis=-1)
        else:
            return context_vector

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'return_sequences': self.return_sequences})
        return config


def build_model(input_shape=(15, 6), lstm_units=20, attention_units=20):
    inputs = Input(shape=input_shape)
    lstm_out = layers.LSTM(lstm_units, return_sequences=True)(inputs)
    ctx, attn = AttentionLayer(units=attention_units, return_sequences=True)(lstm_out)
    outputs = layers.Dense(1)(ctx)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

## 2. Load Data & Model

In [ ]:
# >>> CHANGE THIS PATH to match your Kaggle dataset name <<<
DATA = '/kaggle/input/shap-irrigation-data'

print("Loading data...")
X_test = np.load(f'{DATA}/X_test.npy')
y_test = np.load(f'{DATA}/y_test.npy')
background = np.load(f'{DATA}/background_100.npy')
attention_weights_all = np.load(f'{DATA}/attention_weights.npy')
with open(f'{DATA}/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)

print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Background: {background.shape}")
print(f"Attention weights: {attention_weights_all.shape}")
print(f"Features: {feature_names}")

# Load model
print("\nLoading attention-LSTM model...")
model = build_model((15, 6))
try:
    model.load_weights(f'{DATA}/attention_lstm_final.h5')
    print("Loaded weights via load_weights()")
except:
    model = tf.keras.models.load_model(
        f'{DATA}/attention_lstm_final.h5',
        custom_objects={'AttentionLayer': AttentionLayer}
    )
    print("Loaded full model via load_model()")

# Quick sanity check
pred_sample = model.predict(X_test[:5], verbose=0).flatten()
print(f"Sanity check — predictions for 5 samples: {pred_sample}")

## 3. Stratified Sampling — 200 Instances

In [ ]:
N_SAMPLES = 200
np.random.seed(42)

# Extract features for stratification
humidity_vals = X_test[:, -1, 0]     # humidity at t-1 (most recent)
temp_vals = X_test[:, -1, 1]         # temperature at t-1
valve_vals = y_test                    # valve opening (target)

# Create bins
hum_bins = pd.qcut(humidity_vals, q=4, labels=False, duplicates='drop')
temp_bins = pd.qcut(temp_vals, q=3, labels=False, duplicates='drop')
valve_bins = pd.qcut(valve_vals, q=3, labels=False, duplicates='drop')

# Combine into strata
strata = hum_bins * 100 + temp_bins * 10 + valve_bins
unique_strata = np.unique(strata)
print(f"\nStratification: {len(unique_strata)} unique strata from "
      f"{len(np.unique(hum_bins))} hum × {len(np.unique(temp_bins))} temp × {len(np.unique(valve_bins))} valve bins")

# Sample proportionally from each stratum
selected_indices = []
samples_per_stratum = max(1, N_SAMPLES // len(unique_strata))
remaining = N_SAMPLES

for s in unique_strata:
    mask = np.where(strata == s)[0]
    n_take = min(len(mask), samples_per_stratum)
    chosen = np.random.choice(mask, n_take, replace=False)
    selected_indices.extend(chosen)

# If we need more samples, fill randomly from remaining
selected_indices = np.array(selected_indices)
if len(selected_indices) < N_SAMPLES:
    remaining_pool = np.setdiff1d(np.arange(len(X_test)), selected_indices)
    extra = np.random.choice(remaining_pool, N_SAMPLES - len(selected_indices), replace=False)
    selected_indices = np.concatenate([selected_indices, extra])
elif len(selected_indices) > N_SAMPLES:
    selected_indices = np.random.choice(selected_indices, N_SAMPLES, replace=False)

selected_indices = np.sort(selected_indices)
X_shap = X_test[selected_indices]
y_shap = y_test[selected_indices]
print(f"Selected {len(selected_indices)} stratified samples")
print(f"Valve range in sample: [{y_shap.min():.4f}, {y_shap.max():.4f}]")
print(f"Humidity range: [{X_shap[:,-1,0].min():.4f}, {X_shap[:,-1,0].max():.4f}]")

## 4. Compute SHAP Values (KernelExplainer)

In [ ]:
print("\n" + "=" * 60)
print(f"Computing SHAP for {N_SAMPLES} instances (KernelExplainer)")
print("=" * 60)

# Flatten for KernelExplainer
X_shap_flat = X_shap.reshape(N_SAMPLES, -1)        # (200, 90)
bg_flat = background.reshape(background.shape[0], -1)  # (100, 90)

def model_predict_flat(x_flat):
    x_3d = x_flat.reshape(-1, 15, 6)
    return model.predict(x_3d, verbose=0, batch_size=256)

# Initialize explainer with 100 background samples
explainer = shap.KernelExplainer(model_predict_flat, bg_flat)

t0 = time.time()
shap_values_flat = explainer.shap_values(X_shap_flat, nsamples=200, silent=False)
elapsed = time.time() - t0
print(f"\nSHAP computation done in {elapsed:.1f}s ({elapsed/N_SAMPLES:.2f}s per sample)")

# Handle shape: might be (200, 90, 1) or (200, 90)
if shap_values_flat.ndim == 3:
    shap_values_flat = shap_values_flat.squeeze(-1)

# Reshape to (200, 15, 6)
shap_values_3d = shap_values_flat.reshape(N_SAMPLES, 15, 6)
print(f"SHAP values shape: {shap_values_3d.shape}")

## 5. Feature Importance (from 200 samples)

In [ ]:
feature_importance = np.abs(shap_values_3d).mean(axis=(0, 1))
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\n" + "=" * 60)
print("FEATURE IMPORTANCE (200 stratified samples)")
print("=" * 60)
print(importance_df.to_string(index=False))

# Temporal importance
temporal_importance = np.abs(shap_values_3d).mean(axis=0)  # (15, 6)
temporal_df = pd.DataFrame(
    temporal_importance.T,
    index=feature_names,
    columns=[f't-{15-i}' for i in range(15)]
)

print("\nTemporal importance (per-timestep totals):")
ts_totals = temporal_importance.sum(axis=1)
ts_pct = ts_totals / ts_totals.sum() * 100
for t in range(15):
    label = f"t-{15-t}"
    print(f"  {label}: {ts_pct[t]:.1f}%")

last3_pct = ts_pct[-3:].sum()
print(f"\nLast 3 timesteps (t-1 + t-2 + t-3): {last3_pct:.1f}%")

## 6. Bootstrap Resampling (1000 iterations from 200 samples)

In [ ]:
print("\n" + "=" * 60)
print("Bootstrap resampling (1000 iterations)")
print("=" * 60)

N_BOOT = 1000
np.random.seed(42)

boot_feature = np.zeros((N_BOOT, 6))
boot_temporal = np.zeros((N_BOOT, 15, 6))

for b in range(N_BOOT):
    idx = np.random.choice(N_SAMPLES, N_SAMPLES, replace=True)
    boot_shap = shap_values_3d[idx]
    boot_feature[b] = np.abs(boot_shap).mean(axis=(0, 1))
    boot_temporal[b] = np.abs(boot_shap).mean(axis=0)

# CIs
ci_lower = np.percentile(boot_feature, 2.5, axis=0)
ci_upper = np.percentile(boot_feature, 97.5, axis=0)

ci_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean_Importance': boot_feature.mean(axis=0),
    'Std': boot_feature.std(axis=0),
    'CI_Lower': ci_lower,
    'CI_Upper': ci_upper,
    'CI_Width': ci_upper - ci_lower
}).sort_values('Mean_Importance', ascending=False)

print("\nFeature Importance with 95% CIs:")
print(ci_df.to_string(index=False))

# Check ranking stability
rank_stability = {}
for i, feat in enumerate(feature_names):
    ranks = np.array([(np.argsort(-boot_feature[b]).tolist().index(i) + 1) for b in range(N_BOOT)])
    rank_stability[feat] = {
        'mean_rank': ranks.mean(),
        'rank1_pct': (ranks == 1).mean() * 100
    }
print("\nRanking stability:")
for feat, info in sorted(rank_stability.items(), key=lambda x: x[1]['mean_rank']):
    print(f"  {feat:20s}: mean rank {info['mean_rank']:.2f}, ranked #1 in {info['rank1_pct']:.1f}% of iterations")

# SE for top feature
top_feat_idx = importance_df.index[0]  # Index of top feature
top_feat_name = importance_df.iloc[0]['Feature']
top_feat_se = boot_feature[:, feature_names.index(top_feat_name)].std()
print(f"\nSE for {top_feat_name}: {top_feat_se:.6f}")

## 7. SHAP-Attention Convergence

In [ ]:
print("\n" + "=" * 60)
print("SHAP-Attention Convergence")
print("=" * 60)

# SHAP temporal profile (aggregate features per timestep)
shap_temporal_profile = temporal_importance.sum(axis=1)  # (15,)
shap_temporal_norm = shap_temporal_profile / shap_temporal_profile.sum()

# Attention temporal profile (mean across ALL test samples)
attn_temporal_profile = attention_weights_all.mean(axis=0)  # (15,)
# Attention already sums to 1 per sample (softmax), but mean might not sum to exactly 1
attn_temporal_norm = attn_temporal_profile / attn_temporal_profile.sum()

from scipy import stats
r_val, p_val = stats.pearsonr(shap_temporal_norm, attn_temporal_norm)
print(f"Pearson correlation: r = {r_val:.4f}, p = {p_val:.6f}")

# Bootstrap CI for correlation
boot_r = []
for b in range(N_BOOT):
    idx = np.random.choice(N_SAMPLES, N_SAMPLES, replace=True)
    boot_shap_temp = np.abs(shap_values_3d[idx]).mean(axis=0).sum(axis=1)
    boot_shap_norm = boot_shap_temp / boot_shap_temp.sum()
    r_b, _ = stats.pearsonr(boot_shap_norm, attn_temporal_norm)
    boot_r.append(r_b)
boot_r = np.array(boot_r)
r_ci_lower = np.percentile(boot_r, 2.5)
r_ci_upper = np.percentile(boot_r, 97.5)
print(f"95% CI for r: [{r_ci_lower:.4f}, {r_ci_upper:.4f}]")

# Feature-attention correlations per timestep
print("\nFeature-attention correlations per timestep:")
attn_for_shap = attention_weights_all[selected_indices]  # (200, 15)
for f_idx, f_name in enumerate(feature_names[:4]):  # Top 4 features
    feat_vals = X_shap[:, :, f_idx]  # (200, 15)
    corrs = []
    for t in range(15):
        r_ft, _ = stats.pearsonr(feat_vals[:, t], attn_for_shap[:, t])
        corrs.append(r_ft)
    print(f"  {f_name:20s}: t-15={corrs[0]:+.3f}, t-8={corrs[7]:+.3f}, t-5={corrs[10]:+.3f}, "
          f"t-3={corrs[12]:+.3f}, t-1={corrs[14]:+.3f}")

## 8. Figures

In [ ]:
OUT = '/kaggle/working'

# --- Figure 1: Feature Importance with CIs ---
fig, ax = plt.subplots(figsize=(10, 6))
sorted_ci = ci_df.sort_values('Mean_Importance', ascending=True)
y_pos = range(len(sorted_ci))
colors = ['#2196F3' if imp > 0.0005 else '#90CAF9' for imp in sorted_ci['Mean_Importance']]

ax.barh(list(y_pos), sorted_ci['Mean_Importance'], color=colors, alpha=0.85, edgecolor='white')
ax.errorbar(sorted_ci['Mean_Importance'], list(y_pos),
            xerr=[sorted_ci['Mean_Importance'] - sorted_ci['CI_Lower'],
                  sorted_ci['CI_Upper'] - sorted_ci['Mean_Importance']],
            fmt='none', ecolor='black', capsize=5, linewidth=1.5)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(sorted_ci['Feature'], fontsize=12)
ax.set_xlabel('Mean |SHAP Value|', fontsize=13)
ax.set_title('Feature Importance with 95% Bootstrap CIs (n=200, B=1000)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{OUT}/shap_importance_with_ci.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved shap_importance_with_ci.png")

# --- Figure 2: Temporal Importance Heatmap ---
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    temporal_importance.T,
    annot=True, fmt='.5f', cmap='YlOrRd',
    xticklabels=[f't-{15-i}' for i in range(15)],
    yticklabels=feature_names,
    cbar_kws={'label': 'Mean |SHAP Value|'},
    ax=ax
)
ax.set_title('Temporal Feature Importance (200 stratified samples)', fontsize=14, fontweight='bold')
ax.set_xlabel('Timestep', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/temporal_importance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved temporal_importance_heatmap.png")

# --- Figure 3: Waterfall Plots (5 representative samples) ---
# Pick samples spread across the valve range
valve_sorted_idx = np.argsort(y_shap)
waterfall_local = [valve_sorted_idx[i] for i in [0, N_SAMPLES//4, N_SAMPLES//2, 3*N_SAMPLES//4, N_SAMPLES-1]]

expanded_feature_names = []
for t in range(15):
    for f in feature_names:
        expanded_feature_names.append(f"{f}_t-{15-t}")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes.flatten()

for plot_idx, local_idx in enumerate(waterfall_local):
    if plot_idx >= 5:
        break
    plt.sca(axes_flat[plot_idx])
    explanation = shap.Explanation(
        values=shap_values_flat[local_idx],
        base_values=float(explainer.expected_value) if np.isscalar(explainer.expected_value) else float(explainer.expected_value[0]),
        data=X_shap_flat[local_idx],
        feature_names=expanded_feature_names
    )
    shap.waterfall_plot(explanation, max_display=15, show=False)
    pred_val = model.predict(X_shap[local_idx:local_idx+1], verbose=0).flatten()[0]
    axes_flat[plot_idx].set_title(f'Sample {selected_indices[local_idx]} (pred={pred_val:.3f})', fontsize=11)

axes_flat[5].axis('off')
plt.suptitle("SHAP Waterfall Plots — 5 Representative Samples", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/shap_waterfall_plots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved shap_waterfall_plots.png")

## 9. Save All Results

In [ ]:
print("\n" + "=" * 60)
print("Saving results...")
print("=" * 60)

np.save(f'{OUT}/shap_values_200.npy', shap_values_3d)
np.save(f'{OUT}/shap_indices_200.npy', selected_indices)
np.save(f'{OUT}/bootstrap_feature_importance_200.npy', boot_feature)
np.save(f'{OUT}/bootstrap_temporal_importance_200.npy', boot_temporal)

importance_df.to_csv(f'{OUT}/shap_feature_importance_200.csv', index=False)
temporal_df.to_csv(f'{OUT}/shap_temporal_importance_200.csv')
ci_df.to_csv(f'{OUT}/shap_importance_with_ci_200.csv', index=False)

convergence = {
    'pearson_r': float(r_val),
    'pearson_p': float(p_val),
    'r_ci_lower': float(r_ci_lower),
    'r_ci_upper': float(r_ci_upper),
    'n_shap_samples': int(N_SAMPLES),
    'n_bootstrap': int(N_BOOT),
    'shap_temporal_profile': shap_temporal_norm.tolist(),
    'attn_temporal_profile': attn_temporal_norm.tolist(),
    'feature_importance': {fn: float(v) for fn, v in zip(feature_names, feature_importance)},
    'top_feature_se': float(top_feat_se),
    'last3_timestep_pct': float(last3_pct)
}
with open(f'{OUT}/convergence_results_200.json', 'w') as f:
    json.dump(convergence, f, indent=2)

print("\nSaved files in /kaggle/working/:")
for f in sorted(os.listdir(OUT)):
    if f.startswith('shap_') or f.startswith('bootstrap_') or f.startswith('convergence') or f.startswith('temporal'):
        sz = os.path.getsize(os.path.join(OUT, f))
        print(f"  {f:45s} {sz/1024:8.1f} KB")

print("\n" + "=" * 60)
print("DONE! Download all files from /kaggle/working/")
print("=" * 60)
print(f"\nKey results:")
print(f"  Top feature: {importance_df.iloc[0]['Feature']} = {importance_df.iloc[0]['Importance']:.6f}")
print(f"  SHAP-Attention r = {r_val:.4f} [{r_ci_lower:.4f}, {r_ci_upper:.4f}]")
print(f"  Last 3 timesteps: {last3_pct:.1f}%")
print(f"  Top feature SE: {top_feat_se:.6f}")